# Projet final : les récoltes agricoles

**Équipe :** (nom de l'équipe et des 5 membres)

Une direction départementale de l'agriculture (fictive) vous confie trois ans de récoltes, de 2023 à 2025, par saison (printemps, automne), département et culture, avec la pluie tombée pendant chaque saison. Le ministère veut savoir quelles cultures pèsent le plus, où les rendements sont les meilleurs, et si la pluie explique les mauvaises années.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/recoltes_brut.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

On regarde d'abord la taille du fichier et le type de chaque colonne.

In [2]:
print("Lignes, colonnes :", brut.shape)
brut.dtypes

Lignes, colonnes : (408, 6)


annee           int64
saison            str
departement       str
culture           str
surface_ha        str
production_t      str
dtype: object

`surface_ha` et `production_t` devraient être des nombres, or pandas les lit comme du texte (`str`) : certaines cellules contiennent donc autre chose que des chiffres. On cherche lesquelles.

In [3]:
for colonne in ["surface_ha", "production_t"]:
    valeurs = brut[colonne].dropna()
    pas_nombre = valeurs[pd.to_numeric(valeurs, errors="coerce").isna()]
    print(f"{colonne} : {len(pas_nombre)} valeurs qui ne sont pas des nombres, par exemple {list(pas_nombre.unique()[:6])}")

surface_ha : 22 valeurs qui ne sont pas des nombres, par exemple ['2 517', '541 ha', '1 127', '1 185', '1 748', '492 ha']
production_t : 50 valeurs qui ne sont pas des nombres, par exemple ['3753 t', '2299 t', '1306 t', '2394 t', '599 t', '767 t']


Valeurs manquantes et lignes en double :

In [4]:
print("Valeurs manquantes par colonne :")
print(brut.isna().sum())
print()
print("Lignes en double :", brut.duplicated().sum())

Valeurs manquantes par colonne :
annee            0
saison           0
departement      0
culture          0
surface_ha       0
production_t    15
dtype: int64

Lignes en double : 18


Toutes les écritures de chaque colonne de texte. Il devrait y avoir 2 saisons, 10 départements et 7 cultures.

In [5]:
for colonne in ["saison", "departement", "culture"]:
    ecritures = brut[colonne].unique()
    print(f"{colonne} : {len(ecritures)} écritures différentes")
    print(sorted(ecritures, key=str.lower))
    print()

saison : 2 écritures différentes
['Automne', 'Printemps']

departement : 44 écritures différentes
[' Artibonite', ' Centre', " Grand'Anse", ' Nord', ' Nord-Est', ' Nord-Ouest', ' Ouest', ' Sud', ' Sud-Est', 'Artibonite', 'ARTIBONITE', 'artibonite', 'Centre', 'centre', "Grand'Anse", "grand'anse", "GRAND'ANSE", "Grand'Anse ", 'Nippes', 'NIPPES', 'Nippes ', 'Nord', 'NORD', 'nord', 'Nord ', 'Nord-Est', 'nord-est', 'NORD-EST', 'Nord-Est ', 'NORD-OUEST', 'Nord-Ouest', 'nord-ouest', 'Nord-Ouest ', 'Ouest', 'ouest', 'OUEST', 'Ouest ', 'Sud', 'SUD', 'sud', 'Sud ', 'Sud-Est', 'sud-est', 'Sud-Est ']

culture : 33 écritures différentes
[' Banane', ' Haricot', ' Igname', ' Maïs', ' Pois congo', ' Riz', ' Sorgho', 'banane', 'Banane', 'BANANE', 'Banane ', 'Haricot', 'HARICOT', 'haricot', 'Haricot ', 'Igname', 'IGNAME', 'igname', 'Igname ', 'maïs', 'Maïs', 'MAÏS', 'Maïs ', 'Pois congo', 'POIS CONGO', 'pois congo', 'Pois congo ', 'Riz', 'Riz ', 'Sorgho', 'SORGHO', 'sorgho', 'Sorgho ']



In [6]:
print("Années :", sorted(brut["annee"].unique()))

Années : [np.int64(2023), np.int64(2024), np.int64(2025)]


**Problèmes trouvés**, à corriger à l'étape 2 :

1. **Nombres écrits comme du texte** : 22 surfaces et 50 productions contiennent une unité (`"541 ha"`, `"3753 t"`) ou un espace entre les milliers (`"2 517"`).
2. **Écritures multiples** : 44 écritures pour 10 départements et 33 pour 7 cultures, à cause d'espaces avant ou après (`" Maïs"`, `"Ouest "`) et de majuscules (`"NORD-OUEST"`, `"pois congo"`).
3. **Valeurs manquantes** : 15 productions sont vides.
4. **Doublons** : 18 lignes sont répétées à l'identique.

Les colonnes `annee` (3 valeurs) et `saison` (2 valeurs) sont déjà propres.

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(376, 6)`, sans valeur manquante ni doublon, avec 2 valeurs dans `saison`, 10 valeurs dans `departement`, 7 valeurs dans `culture`.

In [7]:
propre = brut.copy()
print("Lignes au départ :", len(propre))

Lignes au départ : 408


**Problème 1 : les nombres écrits comme du texte.** On retire l'unité et les espaces, puis on convertit en nombre. La production reste décimale (`float`) tant qu'il y a des valeurs manquantes.

In [8]:
for colonne in ["surface_ha", "production_t"]:
    propre[colonne] = (
        propre[colonne]
        .str.replace(" ha", "")
        .str.replace(" t", "")
        .str.replace(" ", "")
    )
    propre[colonne] = pd.to_numeric(propre[colonne])

propre[["surface_ha", "production_t"]].describe()

,surface_ha,production_t
count,408.000000,393.000000
mean,1213.852941,3494.437659
std,864.128299,4516.396156
min,150.000000,103.000000
25%,573.750000,639.000000
50%,1017.500000,1489.000000
75%,1587.500000,4815.000000
max,4065.000000,24133.000000


Aucune erreur de conversion, et pas de valeur négative ni nulle : les minimums sont positifs.

**Problème 2 : les écritures multiples.** On enlève les espaces, on passe en minuscules, puis on remplace chaque nom par son écriture officielle, celle des fichiers `departements.csv` et `cultures.csv`. `.str.capitalize()` ne suffirait pas : il donnerait `"Nord-ouest"` et `"Grand'anse"`.

In [9]:
noms_departements = pd.read_csv("donnees/departements.csv")["departement"]
noms_cultures = pd.read_csv("donnees/cultures.csv")["culture"]

officiel_departement = {nom.lower(): nom for nom in noms_departements}
officiel_culture = {nom.lower(): nom for nom in noms_cultures}

propre["saison"] = propre["saison"].str.strip()
propre["departement"] = propre["departement"].str.strip().str.lower().map(officiel_departement)
propre["culture"] = propre["culture"].str.strip().str.lower().map(officiel_culture)

# Un nom inconnu deviendrait une valeur manquante : on vérifie qu'il n'y en a aucun.
print("Noms non reconnus :", propre["departement"].isna().sum() + propre["culture"].isna().sum())
print(propre["departement"].unique())
print(propre["culture"].unique())

Noms non reconnus : 0
<ArrowStringArray>
[     'Ouest', 'Artibonite',       'Nord',        'Sud',     'Centre',
 'Nord-Ouest',    'Sud-Est', 'Grand'Anse',   'Nord-Est',     'Nippes']
Length: 10, dtype: str
<ArrowStringArray>
['Maïs', 'Sorgho', 'Riz', 'Haricot', 'Pois congo', 'Banane', 'Igname']
Length: 7, dtype: str


**Problème 3 : les doublons.** On les supprime *après* avoir uniformisé les noms : `"Ouest"` et `"Ouest "` ne sont reconnus comme identiques qu'une fois les espaces retirés. Garder un doublon compterait deux fois la même récolte.

In [10]:
avant = len(propre)
propre = propre.drop_duplicates()
print(f"Doublons supprimés : {avant - len(propre)} ({avant} -> {len(propre)} lignes)")

Doublons supprimés : 18 (408 -> 390 lignes)


**Problème 4 : les productions manquantes.** Sans production, on ne peut calculer ni rendement ni valeur. Remplacer le vide par une moyenne inventerait une récolte : on supprime ces lignes.

In [11]:
avant = len(propre)
propre = propre.dropna(subset=["production_t"])
propre["production_t"] = propre["production_t"].astype(int)
propre = propre.reset_index(drop=True)
print(f"Lignes sans production supprimées : {avant - len(propre)} ({avant} -> {len(propre)} lignes)")

Lignes sans production supprimées : 14 (390 -> 376 lignes)


**Contrôle.** Chaque `assert` arrête le notebook si le nettoyage est faux.

In [12]:
assert propre.shape == (376, 6)
assert propre.isna().sum().sum() == 0
assert propre.duplicated().sum() == 0
assert propre["saison"].nunique() == 2
assert propre["departement"].nunique() == 10
assert propre["culture"].nunique() == 7
# Une seule ligne par année, saison, département et culture :
assert not propre.duplicated(["annee", "saison", "departement", "culture"]).any()

print("Tous les contrôles passent :", propre.shape)
propre.dtypes

Tous les contrôles passent : (376, 6)


annee           int64
saison            str
departement       str
culture           str
surface_ha      int64
production_t    int64
dtype: object

**Vérification de vraisemblance.** On compare le rendement de chaque ligne au rendement de référence de sa culture, pour repérer une valeur aberrante (une faute de frappe qui multiplierait la production par 10, par exemple).

In [13]:
reference = pd.read_csv("donnees/cultures.csv").set_index("culture")["rendement_moyen_t_ha"]
ecart = propre["production_t"] / propre["surface_ha"] / propre["culture"].map(reference)
print(f"Rendement / référence : de {ecart.min():.2f} à {ecart.max():.2f}")

Rendement / référence : de 0.64 à 1.82


Les rendements vont de la moitié à moins de deux fois la référence : c'est la variation normale entre bonnes et mauvaises saisons, aucune ligne n'est aberrante.

Il manque des combinaisons : 10 départements × 6 cultures × 6 saisons, plus le riz cultivé dans 5 départements, font 390 combinaisons possibles. Il en reste 376 : les 14 manquantes sont les productions vides. On s'en souviendra quand on additionnera les tonnes par année.

## 3. Jointures

Joignez `propre` avec `cultures.csv`, `pluies.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 376 lignes après les jointures, autant qu'avant.

Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/recoltes_propre.csv", index=False)
```

## 4. Quelles cultures pèsent le plus, en tonnes puis en valeur (gourdes) ? Le classement change-t-il ?

**Ce qu'on en conclut :**

## 5. Où les rendements (tonnes par hectare) sont-ils les meilleurs, culture par culture ?

**Ce qu'on en conclut :**

## 6. La pluie explique-t-elle les rendements ? Pour quelles cultures ? Montrez-le avec des nuages de points.

**Ce qu'on en conclut :**

## 7. Quelles cultures progressent ou reculent entre 2023 et 2025 ?

**Ce qu'on en conclut :**

## 8. Rapportez la production à la population de chaque département : quels départements produisent le plus par habitant ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.